# Nockasm

[`nockasm`](https://github.com/sigilante/nockasm) is a thin Python macro layer over canonical Nock, designed to make hand-written Nock more legible.  It is not a separate language or a compiler; every `nockasm` expression expands to a plain Nock noun that you can paste directly into any Nock evaluator.  The primary audience is people learning to write Nock by hand or verifying that a piece of Nock does what they think it does.  A secondary audience could be designers of Nock-based languages or compilers who want a more legible way to express the formulas they are generating.

`nockasm` is available on [PyPI](https://pypi.org/project/nockasm/).  This notebook also uses [`pinochle`](https://github.com/sigilante/pinochle), a Nock evaluator for Python, to run the expanded formulas inline.

In [ ]:
!pip install nockasm pinochle

In [ ]:
from nockasm import expand
from pinochle import nock, parse_noun

## Named opcodes

Raw Nock opcodes are bare numbers.  `nockasm` replaces them with named forms:

| `nockasm`        | Canonical Nock | Meaning                        |
|------------------|----------------|--------------------------------|
| `(%slot N)`      | `[0 N]`        | address slot N in subject      |
| `(%const X)`     | `[1 X]`        | produce literal X              |
| `(%eval F G)`    | `[2 F G]`      | evaluate F against G           |
| `(%isa F)`       | `[3 F]`        | test cell/atom                 |
| `(%inc F)`       | `[4 F]`        | increment atom                 |
| `(%eq F G)`      | `[5 F G]`      | test equality                  |
| `(%if C T E)`    | `[6 C T E]`    | conditional branch             |
| `(%comp F G)`    | `[7 F G]`      | compose formulas               |
| `(%push F G)`    | `[8 F G]`      | push F onto subject, eval G    |
| `(%call N F)`    | `[9 N F]`      | call arm at slot N             |
| `(%edit N V F)`  | `[10 [N V] F]` | edit slot N of F with V        |
| `(%hint T F)`    | `[11 T F]`     | static hint                    |
| `(%hintd T C F)` | `[11 [T C] F]` | dynamic hint with clue         |

Axis aliases name the standard Hoon core slots: `(%self)` → `[0 1]`, `(%battery)` → `[0 2]`, `(%payload)` → `[0 3]`, `(%sample)` → `[0 6]`, `(%context)` → `[0 7]`.

In [ ]:
# Increment the subject (self)
expand("(%inc (%self))")

In [ ]:
# Test equality of slots 2 and 3
expand("(%eq (%slot 2) (%slot 3))")

In [ ]:
# Conditional: if slot 2 equals 0, return 1, else return slot 2
expand("(%if (%eq (%slot 2) (%const 0)) (%const 1) (%slot 2))")

## Axis schemas

`:subject {.a .b .c}` binds names to the standard right-leaning binary tree addresses (axis 2, 6, 7 for a three-element subject, following Hoon convention).  After the declaration, `.a`, `.b`, `.c` expand to `[0 2]`, `[0 6]`, `[0 7]` respectively.

The tree layout for a two-element subject `[x y]`:

```
    1
   / \
  2   3
 (x) (y)
```

For three elements `[x y z]` (right-leaning: `[x [y z]]`):

```
    1
   / \
  2   3
 (x) / \
    6   7
   (y) (z)
```

In [ ]:
# Two-element subject: .x at axis 2, .y at axis 3
expand("""
:subject {.x .y}
(%eq .x .y)
""")

In [ ]:
# Three-element subject: .target at axis 6
expand("""
:subject {.before .target .after}
(%inc .target)
""")

## `#let .name = VALUE in BODY`

Pushes `VALUE` onto the subject via opcode 8, binding `.name` to the new head (axis 2).  Existing schema names shift rightward through `peg(3, axis)` so they remain valid in `BODY`.

This is the `nockasm` equivalent of a local `let` binding: it evaluates `VALUE` against the current subject and pushes the result as a new head, making it available under `.name` while the body runs.

In [ ]:
# Expand: bind .next to the incremented .target, then return all three values
expand("""
:subject {.before .target .after}
#let .next = (%inc .target) in
  [.before .next .after]
""")

In [ ]:
# Run it: subject [10 41 99] -> [10 42 99]
formula = parse_noun(expand("""
:subject {.before .target .after}
#let .next = (%inc .target) in
  [.before .next .after]
"""))

nock(parse_noun("[10 41 99]"), formula)

## `#match EXPR { PAT => BODY ... _ => DEFAULT }`

Evaluates `EXPR` once via opcode 8, then dispatches on its value using nested opcode-6 branches.  The `_ =>` default branch is required.  Patterns are noun literals compared against the scrutinee at runtime.

Each pattern arm compiles to a nested `[6 [5 [1 PAT] 0 2] BODY ...]` chain, so a three-arm match with a default produces two nested `if` branches sharing a single evaluation of the scrutinee.

In [ ]:
# Expand: dispatch on .tag
expand("""
:subject {.tag .data}
#match .tag {
  1 => (%inc .data)
  2 => .data
  _ => 0
}
""")

In [ ]:
# Run it across all three cases
formula = parse_noun(expand("""
:subject {.tag .data}
#match .tag {
  1 => (%inc .data)
  2 => .data
  _ => 0
}
"""))

print(nock(parse_noun("[1 41]"), formula))  # tag=1: increment .data -> 42
print(nock(parse_noun("[2 41]"), formula))  # tag=2: return .data -> 41
print(nock(parse_noun("[9 41]"), formula))  # tag=9: default -> 0

## Combining features

The macros compose freely.  Here is a small decrement-style program: given a subject `[n acc]`, match on `n` and either return `acc` when `n` is zero or recurse (expressed as `#let` + `#match`) incrementing `acc`.

In [ ]:
# Nested let + match: compute absolute difference between two slots
src = """
:subject {.a .b}
#let .eq = (%eq .a .b) in
#match .eq {
  0 => (%inc .a)
  _ => .b
}
"""
print(expand(src))

## Relation to canonical Nock

`nockasm` is a preprocessor, not a language.  Every expression it accepts expands to a noun that is bit-identical to what you would write by hand.  There is no runtime, no separate VM, and no overhead beyond the Python expansion step itself.  The benchmark programs bundled with the library (`dec`, `add`, `factorial`, `fibonacci`, `ackermann`) each expand to nouns bit-identical to the corresponding reference formulas from the Urbit benchmark suite.

:::{note}
See [The Assembly Language Approach](../understanding/assembly-language-approach.ipynb) for background on the relationship between Nock and conventional assembly language, and why `nockasm` occupies a similar role to a hardware assembler—without being tied to any machine architecture.
:::

## CLI

Outside of notebooks, `nockasm` can be driven from the shell:

```bash
python -m nockasm program.nasm           # canonical flat Nock
python -m nockasm --pretty program.nasm  # explicit binary cells
echo "(%inc (%self))" | python -m nockasm
```

The CLI reads `.nasm` source files or stdin and writes the expanded noun to stdout, making it easy to pipe into any Nock evaluator or use in a build pipeline.